# Tasks
1. Download a list of all maps from mapexplorer via API in JSON. This needs to be done in loops to allow for pagination. Concatenate the JSON records and save to file `all_data.json`
2. Convert the json file to a pandas dataframe and save as a csv with relevant attributes pulled out as their own columns.
3. Iterate through all maps in the all_data JSON file (indicated by variable `id`. Download the image (.tif) and CSV (.csv) files for each map from the `all_data` list. For each item, extract its `id`, construct the respective download URLs using
  -  `https://mapexplorer.mcmaster.ca/maps/{id}/export.tif` and
  -  `https://mapexplorer.mcmaster.ca/maps/{id}/gcps.csv`, and save the files locally using the `id` in their filenames (e.g., `1137.tif`, `1137.csv`).
4. Finally, list the names of the downloaded files to verify the process.



In [ ]:
import requests
import json
import pandas as pd

# Re-initialize and populate all_data as it was not defined in the current context
all_data = []
base_url = "https://mapexplorer.mcmaster.ca/api/v1/maps.json"
download_path = "E:/Users/brodeujj/mapwarper/"

print("Re-downloading all_data to ensure its availability...")
for page_num in range(1, 7):  # Loop from page 1 to 6
    url = f"{base_url}?page={page_num}&per_page=500"
    try:
        response = requests.get(url)
        response.raise_for_status()
        page_data = response.json()

        if isinstance(page_data, dict) and 'data' in page_data and isinstance(page_data['data'], list):
            all_data.extend(page_data['data'])
        else:
            print(f"Warning: Page {page_num} did not contain expected 'data' list during re-download.")

    except requests.exceptions.RequestException as e:
        print(f"Error re-downloading data from page {page_num}: {e}")

print(f"all_data re-populated with {len(all_data)} items.")

# Save the json file:
json_file_name = "all_data.json"
with open(download_path+json_file_name, 'w') as f:
    json.dump(all_data, f, indent=4)
print(f"Data successfully saved to {json_file_name}")

# Convert all_data to a pandas data frame and save it as a csv
df_all_data = pd.DataFrame(all_data)
print("DataFrame created successfully.")
print(df_all_data.head())

# Extract the 'title' and 'source_uri' from the 'attributes' column
df_all_data['title'] = df_all_data['attributes'].apply(lambda x: x.get('title'))
df_all_data['source_uri'] = df_all_data['attributes'].apply(lambda x: x.get('source_uri'))
df_all_data['unique_id'] = df_all_data['attributes'].apply(lambda x: x.get('unique_id'))
df_all_data['date_depicted'] = df_all_data['attributes'].apply(lambda x: x.get('date_depicted'))

print("Extracted 'title' and 'source_uri' columns:")
print(df_all_data[['id', 'title', 'source_uri','unique_id','date_depicted']].head())

csv_file_name = "all_data.csv"
df_all_data.to_csv(download_path+csv_file_name, index=False)
print(f"DataFrame successfully saved to {csv_file_name}")


#################

downloaded_files = []
# Loop through the items of the all_data list
for item in all_data:
    try:
        item_id = item['id']
        print(f"Processing item with ID: {item_id}")

        # Construct URLs
        tif_url = f"https://mapexplorer.mcmaster.ca/maps/{item_id}/export.tif"
        csv_url = f"https://mapexplorer.mcmaster.ca/maps/{item_id}/gcps.csv"

        # Download TIFF image
        print(f"Downloading TIF from: {tif_url}")
        tif_response = requests.get(tif_url, stream=True)
        tif_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        tif_filename = f"{item_id}.tif"
        with open(download_path+tif_filename, 'wb') as f:
            for chunk in tif_response.iter_content(chunk_size=8192):
                f.write(chunk)
        downloaded_files.append(tif_filename)
        print(f"Successfully downloaded {tif_filename}")

        # Download CSV file
        print(f"Downloading CSV from: {csv_url}")
        csv_response = requests.get(csv_url)
        csv_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        csv_filename = f"{item_id}.csv"
        with open(download_path+csv_filename, 'w', encoding='utf-8') as f:
            f.write(csv_response.text)
        downloaded_files.append(csv_filename)
        print(f"Successfully downloaded {csv_filename}")

    except KeyError as e:
        print(f"Error: 'id' key not found in item or other key error: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading files for item ID {item.get('id', 'N/A')}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for item ID {item.get('id', 'N/A')}: {e}")

print(f"\nTotal files downloaded: {len(downloaded_files)}")
print("Downloaded files:", downloaded_files)